In [5]:
# ============================================================
# ROUND 2 QC - INDEPENDENT NOTEBOOK SETUP
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/home/jovyan/project work/data_analyssis/fine tuning"
)

GENERATION_DIR = (
    PROJECT_DIR
    / "outputs"
    / "synthetic_generation"
)

# Round 2 RAW generation files
ROUND2_GENERATION_DIR = (
    GENERATION_DIR
    / "full_round2"
)

# Round 2 QC outputs
ROUND2_QC_OUTPUT_DIR = (
    GENERATION_DIR
    / "qc_pipeline"
    / "round2"
)

ROUND2_QC_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# RUHSOLD LABEL MAPPING
# ============================================================

ID_TO_LABEL = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}

LABEL_TO_ID = {
    label: class_id
    for class_id, label
    in ID_TO_LABEL.items()
}


# ============================================================
# CHECK PATHS
# ============================================================

print(
    "Project directory:"
)
print(PROJECT_DIR)

print(
    "\nGeneration directory:"
)
print(GENERATION_DIR)

print(
    "\nRound 2 generation directory:"
)
print(ROUND2_GENERATION_DIR)

print(
    "\nRound 2 QC directory:"
)
print(ROUND2_QC_OUTPUT_DIR)

print(
    "\nRound 2 generation directory exists:",
    ROUND2_GENERATION_DIR.exists()
)

Project directory:
/home/jovyan/project work/data_analyssis/fine tuning

Generation directory:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation

Round 2 generation directory:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/full_round2

Round 2 QC directory:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2

Round 2 generation directory exists: True


In [2]:
# ============================================================
# LOAD ROUND 2 RAW GENERATION FILES
# ============================================================

round2_files = sorted(
    ROUND2_GENERATION_DIR.glob(
        "*.csv"
    )
)

print(
    "Round 2 batch files found:",
    len(round2_files)
)

for path in round2_files:
    print(path.name)


if len(round2_files) == 0:
    raise RuntimeError(
        "No Round 2 generation files were found."
    )


# ============================================================
# COMBINE ROUND 2 FILES
# ============================================================

round2_frames = []

for path in round2_files:

    batch_df = pd.read_csv(
        path
    )

    batch_df[
        "source_batch_file"
    ] = path.name

    round2_frames.append(
        batch_df
    )


round2_raw_df = pd.concat(
    round2_frames,
    ignore_index=True
)


# ============================================================
# VERIFY ROUND 2
# ============================================================

print(
    "\nCombined Round 2 shape:",
    round2_raw_df.shape
)

print(
    "\nClass distribution:"
)

display(
    round2_raw_df[
        "target_label"
    ]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)


print(
    "\nMissing generated texts:",
    round2_raw_df[
        "generated_text"
    ]
    .isna()
    .sum()
)

print(
    "Duplicate candidate IDs:",
    round2_raw_df[
        "production_candidate_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Exact duplicate generated texts:",
    round2_raw_df[
        "generated_text"
    ]
    .duplicated()
    .sum()
)


# ============================================================
# EXPECTED ROUND 2 COUNTS
# ============================================================

expected_round2_counts = {
    2: 900,
    3: 1300,
    4: 450,
}

print(
    "\nExpected vs actual:"
)

for class_id, expected in (
    expected_round2_counts.items()
):

    actual = int(
        (
            round2_raw_df[
                "class_id"
            ].astype(int)
            == class_id
        ).sum()
    )

    print(
        f"{class_id} - "
        f"{ID_TO_LABEL[class_id]}: "
        f"{actual} / {expected}"
    )

    assert actual == expected


assert len(round2_raw_df) == 2650

assert (
    round2_raw_df[
        "production_candidate_id"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    round2_raw_df[
        "generated_text"
    ]
    .isna()
    .sum()
    == 0
)


print(
    "\n✓ Round 2 raw generation data "
    "loaded and verified successfully."
)

Round 2 batch files found: 15
profane_batch_004.csv
profane_batch_005.csv
profane_batch_006.csv
religious_hate_batch_006.csv
religious_hate_batch_007.csv
religious_hate_batch_008.csv
religious_hate_batch_009.csv
religious_hate_batch_010.csv
sexism_batch_005.csv
sexism_batch_006.csv
sexism_batch_007.csv
sexism_batch_008.csv
sexism_batch_009.csv
sexism_batch_010.csv
sexism_batch_011.csv

Combined Round 2 shape: (2650, 17)

Class distribution:


,Class,Count
0,Sexism,1300
1,Religious Hate,900
2,Profane,450



Missing generated texts: 0
Duplicate candidate IDs: 0
Exact duplicate generated texts: 30

Expected vs actual:
2 - Religious Hate: 900 / 900
3 - Sexism: 1300 / 1300
4 - Profane: 450 / 450

✓ Round 2 raw generation data loaded and verified successfully.


In [3]:
# ============================================================
# STAGE 2
# Load selected embedding model
# ============================================================

import torch
from sentence_transformers import SentenceTransformer

E5_MODEL_NAME = "intfloat/multilingual-e5-large"

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

e5_model = SentenceTransformer(
    E5_MODEL_NAME,
    device=device
)

print("Loaded:", E5_MODEL_NAME)
print("Device:", device)

[HAMI-core Msg(131:140540241079616:libvgpu.c:839)]: Initializing.....
[HAMI-core Msg(131:140540241079616:libvgpu.c:855)]: Initialized


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loaded: intfloat/multilingual-e5-large
Device: cuda


In [6]:
# ============================================================
# ROUND 2 QC
# STAGE 0 - BASIC QUALITY CONTROL
# ============================================================

import re
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

ROUND2_QC_OUTPUT_DIR = (
    GENERATION_DIR
    / "qc_pipeline"
    / "round2"
)

ROUND2_QC_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FROZEN STAGE 0 SETTINGS
# ============================================================

MIN_WORDS = 2
MAX_WORDS = 70


def contains_urdu_script(text):
    """
    Return True if Arabic/Urdu-script characters are present.
    """
    return bool(
        re.search(
            r"[\u0600-\u06FF]",
            str(text)
        )
    )


def has_latin_content(text):
    """
    Require at least one Latin letter.
    """
    return bool(
        re.search(
            r"[A-Za-z]",
            str(text)
        )
    )


def count_words(text):
    """
    Count whitespace-separated tokens.
    """
    return len(
        str(text)
        .strip()
        .split()
    )


print(
    "Round 2 Stage 0 helper functions loaded."
)


# ============================================================
# CALCULATE BASIC QUALITY FLAGS
# ============================================================

stage0_df = (
    round2_raw_df
    .copy()
    .reset_index(drop=True)
)

stage0_df[
    "generated_text"
] = (
    stage0_df[
        "generated_text"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


stage0_df[
    "word_count"
] = (
    stage0_df[
        "generated_text"
    ]
    .apply(
        count_words
    )
)


stage0_df[
    "is_empty"
] = (
    stage0_df[
        "generated_text"
    ] == ""
)


stage0_df[
    "has_urdu_script"
] = (
    stage0_df[
        "generated_text"
    ]
    .apply(
        contains_urdu_script
    )
)


stage0_df[
    "has_latin_content"
] = (
    stage0_df[
        "generated_text"
    ]
    .apply(
        has_latin_content
    )
)


stage0_df[
    "is_non_linguistic"
] = (
    ~stage0_df[
        "has_latin_content"
    ]
)


stage0_df[
    "is_too_short"
] = (
    stage0_df[
        "word_count"
    ]
    < MIN_WORDS
)


stage0_df[
    "is_too_long"
] = (
    stage0_df[
        "word_count"
    ]
    > MAX_WORDS
)


stage0_df[
    "is_exact_duplicate"
] = (
    stage0_df[
        "generated_text"
    ]
    .duplicated(
        keep="first"
    )
)


# ============================================================
# ASSIGN REJECTION REASONS
# ============================================================

def get_stage0_rejection_reason(row):

    reasons = []

    if row[
        "is_empty"
    ]:
        reasons.append(
            "Empty text"
        )

    if row[
        "has_urdu_script"
    ]:
        reasons.append(
            "Urdu script"
        )

    if row[
        "is_non_linguistic"
    ]:
        reasons.append(
            "No Latin text"
        )

    if row[
        "is_too_short"
    ]:
        reasons.append(
            "Too short"
        )

    if row[
        "is_too_long"
    ]:
        reasons.append(
            "Too long"
        )

    if row[
        "is_exact_duplicate"
    ]:
        reasons.append(
            "Exact duplicate"
        )

    if reasons:
        return "; ".join(
            reasons
        )

    return pd.NA


stage0_df[
    "stage0_rejection_reason"
] = (
    stage0_df.apply(
        get_stage0_rejection_reason,
        axis=1
    )
)


stage0_df[
    "stage0_pass"
] = (
    stage0_df[
        "stage0_rejection_reason"
    ]
    .isna()
)


# ============================================================
# CREATE PASSED / REJECTED SETS
# ============================================================

stage0_passed_df = (
    stage0_df[
        stage0_df[
            "stage0_pass"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


stage0_rejected_df = (
    stage0_df[
        ~stage0_df[
            "stage0_pass"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print(
    "ROUND 2 - STAGE 0 BASIC QC"
)
print("=" * 60)

print(
    "Input samples:",
    len(stage0_df)
)

print(
    "Passed:",
    len(stage0_passed_df)
)

print(
    "Rejected:",
    len(stage0_rejected_df)
)

print(
    "Pass rate:",
    f"{len(stage0_passed_df) / len(stage0_df):.2%}"
)


# ============================================================
# CLASS-WISE SUMMARY
# ============================================================

stage0_class_summary = (
    stage0_df
    .groupby(
        [
            "class_id",
            "target_label",
        ],
        dropna=False
    )
    .agg(
        input_samples=(
            "generated_text",
            "size"
        ),
        passed=(
            "stage0_pass",
            "sum"
        ),
    )
    .reset_index()
)


stage0_class_summary[
    "rejected"
] = (
    stage0_class_summary[
        "input_samples"
    ]
    -
    stage0_class_summary[
        "passed"
    ]
)


stage0_class_summary[
    "pass_rate"
] = (
    stage0_class_summary[
        "passed"
    ]
    /
    stage0_class_summary[
        "input_samples"
    ]
)


print(
    "\nClass-wise Stage 0 results:"
)

display(
    stage0_class_summary.round(4)
)


print(
    "\nRejection reasons:"
)

display(
    stage0_rejected_df[
        "stage0_rejection_reason"
    ]
    .value_counts()
    .rename_axis("Reason")
    .reset_index(name="Count")
)


# ============================================================
# SAVE OUTPUTS
# ============================================================

STAGE0_AUDIT_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage0_audit.csv"
)

STAGE0_PASSED_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage0_passed.csv"
)

STAGE0_REJECTED_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage0_rejected.csv"
)


stage0_df.to_csv(
    STAGE0_AUDIT_PATH,
    index=False,
    encoding="utf-8"
)

stage0_passed_df.to_csv(
    STAGE0_PASSED_PATH,
    index=False,
    encoding="utf-8"
)

stage0_rejected_df.to_csv(
    STAGE0_REJECTED_PATH,
    index=False,
    encoding="utf-8"
)


print(
    "\nRound 2 Stage 0 outputs saved."
)

print(
    "Audit:",
    STAGE0_AUDIT_PATH
)

print(
    "Passed:",
    STAGE0_PASSED_PATH
)

print(
    "Rejected:",
    STAGE0_REJECTED_PATH
)

print(
    "\nSamples passed to Stage 1:",
    len(stage0_passed_df)
)

Round 2 Stage 0 helper functions loaded.

ROUND 2 - STAGE 0 BASIC QC
Input samples: 2650
Passed: 2603
Rejected: 47
Pass rate: 98.23%

Class-wise Stage 0 results:


,class_id,target_label,input_samples,passed,rejected,pass_rate
0,2,Religious Hate,900,888,12,0.9867
1,3,Sexism,1300,1269,31,0.9762
2,4,Profane,450,446,4,0.9911



Rejection reasons:


,Reason,Count
0,Exact duplicate,24
1,Too short,10
2,No Latin text; Too short,7
3,Too short; Exact duplicate,6



Round 2 Stage 0 outputs saved.
Audit: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage0_audit.csv
Passed: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage0_passed.csv
Rejected: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage0_rejected.csv

Samples passed to Stage 1: 2603


In [7]:
# ============================================================
# STAGE 1
# LOAD TUNED XLM-R LABEL-CONSISTENCY CLASSIFIER
# ============================================================

import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)


XLMR_CHECKPOINT = (
    PROJECT_ROOT
    / "classifier"
    / "outputs"
    / "xlm_roberta_tuned"
    / "checkpoint-1203"
)

print(
    "Checkpoint exists:",
    XLMR_CHECKPOINT.exists()
)


xlmr_tokenizer = AutoTokenizer.from_pretrained(
    XLMR_CHECKPOINT
)

xlmr_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        XLMR_CHECKPOINT
    )
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

xlmr_model.to(device)
xlmr_model.eval()

print("Device:", device)
print("Model loaded successfully.")

[HAMI-core Msg(173:139805766655296:libvgpu.c:839)]: Initializing.....


Checkpoint exists: True


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[HAMI-core Msg(173:139805766655296:libvgpu.c:855)]: Initialized


Device: cuda
Model loaded successfully.


In [8]:
# ============================================================
# STAGE 1
# RUHSOLD LABEL MAPPING
# ============================================================

ID_TO_LABEL = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}

LABEL_TO_ID = {
    label: idx
    for idx, label in ID_TO_LABEL.items()
}

print(ID_TO_LABEL)

{0: 'Abusive/Offensive', 1: 'Normal', 2: 'Religious Hate', 3: 'Sexism', 4: 'Profane'}


In [9]:
# ============================================================
# STAGE 1
# XLM-R PREDICTION FUNCTION
# ============================================================

def predict_xlmr_class(
    texts,
    batch_size=32,
    max_length=128,
):
    """
    Predict RUHSOLD class IDs for a list of texts.
    """

    all_predictions = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = texts[
            start:start + batch_size
        ]

        encoded = xlmr_tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.no_grad():

            outputs = xlmr_model(
                **encoded
            )

        batch_predictions = (
            outputs.logits
            .argmax(dim=-1)
            .cpu()
            .tolist()
        )

        all_predictions.extend(
            batch_predictions
        )

    return all_predictions


print("XLM-R prediction function loaded.")

XLM-R prediction function loaded.


In [10]:
# ============================================================
# ROUND 2 QC
# STAGE 1 - XLM-R LABEL CONSISTENCY FILTER
# ============================================================

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)


# ============================================================
# LOAD STAGE 0 SURVIVORS
# ============================================================

stage1_input_df = pd.read_csv(
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage0_passed.csv"
)

print(
    "Round 2 Stage 1 input samples:",
    len(stage1_input_df)
)

print(
    "\nClass distribution:"
)

display(
    stage1_input_df[
        "target_label"
    ]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)


# ============================================================
# LOAD FROZEN XLM-R LABEL-CONSISTENCY CLASSIFIER
# ============================================================

XLMR_CHECKPOINT = (
    Path(
        "/home/jovyan/project work/data_analyssis"
    )
    / "classifier"
    / "outputs"
    / "xlm_roberta_tuned"
    / "checkpoint-1203"
)

print(
    "\nXLM-R checkpoint:"
)

print(
    XLMR_CHECKPOINT
)

print(
    "Checkpoint exists:",
    XLMR_CHECKPOINT.exists()
)


xlmr_tokenizer = (
    AutoTokenizer.from_pretrained(
        XLMR_CHECKPOINT
    )
)

xlmr_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        XLMR_CHECKPOINT
    )
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

xlmr_model.to(
    device
)

xlmr_model.eval()

print(
    "Device:",
    device
)

print(
    "Frozen XLM-R verifier loaded successfully."
)


# ============================================================
# RUHSOLD LABEL MAPPING
# ============================================================

ID_TO_LABEL = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}


# ============================================================
# XLM-R PREDICTION FUNCTION
# ============================================================

def predict_xlmr_class(
    texts,
    batch_size=32,
    max_length=128,
):
    """
    Predict RUHSOLD class IDs for a list of texts
    using the frozen tuned XLM-R classifier.
    """

    all_predictions = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = texts[
            start:start + batch_size
        ]

        encoded = xlmr_tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(device)
            for key, value
            in encoded.items()
        }

        with torch.no_grad():

            outputs = (
                xlmr_model(
                    **encoded
                )
            )

        batch_predictions = (
            outputs.logits
            .argmax(dim=-1)
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            batch_predictions
        )

    return all_predictions


print(
    "Prediction function loaded."
)


# ============================================================
# RUN LABEL-CONSISTENCY VERIFICATION
# ============================================================

stage1_df = (
    stage1_input_df
    .copy()
    .reset_index(drop=True)
)


texts = (
    stage1_df[
        "generated_text"
    ]
    .fillna("")
    .astype(str)
    .tolist()
)


stage1_df[
    "xlmr_predicted_class_id"
] = (
    predict_xlmr_class(
        texts=texts,
        batch_size=32,
        max_length=128,
    )
)


stage1_df[
    "xlmr_predicted_class"
] = (
    stage1_df[
        "xlmr_predicted_class_id"
    ]
    .map(
        ID_TO_LABEL
    )
)


# ============================================================
# APPLY FROZEN STAGE 1 ACCEPTANCE RULE
#
# Accept only if:
# predicted class == intended target class
# ============================================================

stage1_df[
    "stage1_pass"
] = (
    stage1_df[
        "xlmr_predicted_class_id"
    ].astype(int)
    ==
    stage1_df[
        "class_id"
    ].astype(int)
)


stage1_df[
    "stage1_rejection_reason"
] = (
    np.where(
        stage1_df[
            "stage1_pass"
        ],
        pd.NA,
        (
            "Predicted class does not "
            "match intended class"
        ),
    )
)


# ============================================================
# CREATE PASSED / REJECTED DATASETS
# ============================================================

stage1_passed_df = (
    stage1_df[
        stage1_df[
            "stage1_pass"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


stage1_rejected_df = (
    stage1_df[
        ~stage1_df[
            "stage1_pass"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# OVERALL SUMMARY
# ============================================================

print("\n" + "=" * 60)

print(
    "ROUND 2 - STAGE 1 LABEL CONSISTENCY"
)

print("=" * 60)

print(
    "Input samples:",
    len(stage1_df)
)

print(
    "Passed:",
    len(stage1_passed_df)
)

print(
    "Rejected:",
    len(stage1_rejected_df)
)

print(
    "Pass rate:",
    f"{len(stage1_passed_df) / len(stage1_df):.2%}"
)


# ============================================================
# CLASS-WISE SUMMARY
# ============================================================

stage1_class_summary = (
    stage1_df
    .groupby(
        [
            "class_id",
            "target_label",
        ],
        dropna=False
    )
    .agg(
        input_samples=(
            "stage1_pass",
            "size"
        ),
        passed=(
            "stage1_pass",
            "sum"
        ),
    )
    .reset_index()
)


stage1_class_summary[
    "rejected"
] = (
    stage1_class_summary[
        "input_samples"
    ]
    -
    stage1_class_summary[
        "passed"
    ]
)


stage1_class_summary[
    "pass_rate"
] = (
    stage1_class_summary[
        "passed"
    ]
    /
    stage1_class_summary[
        "input_samples"
    ]
)


print(
    "\nClass-wise Stage 1 results:"
)

display(
    stage1_class_summary.round(4)
)


# ============================================================
# TARGET CLASS × XLM-R PREDICTED CLASS
# ============================================================

print(
    "\nTarget class × XLM-R predicted class:"
)

stage1_confusion = pd.crosstab(
    stage1_df[
        "target_label"
    ],
    stage1_df[
        "xlmr_predicted_class"
    ],
    margins=True,
)

display(
    stage1_confusion
)


# ============================================================
# REJECTION PATTERNS
# ============================================================

print(
    "\nRejected samples only: "
    "target class × predicted class"
)

if len(
    stage1_rejected_df
) > 0:

    stage1_rejection_patterns = (
        pd.crosstab(
            stage1_rejected_df[
                "target_label"
            ],
            stage1_rejected_df[
                "xlmr_predicted_class"
            ],
        )
    )

    display(
        stage1_rejection_patterns
    )


# ============================================================
# INSPECT SAMPLE REJECTIONS
# ============================================================

print(
    "\nRejected samples:",
    len(stage1_rejected_df)
)


inspection_columns = [
    "production_candidate_id",
    "class_id",
    "target_label",
    "generated_text",
    "xlmr_predicted_class_id",
    "xlmr_predicted_class",
    "stage1_rejection_reason",
]


available_columns = [
    column
    for column in inspection_columns
    if column in stage1_rejected_df.columns
]


display(
    stage1_rejected_df[
        available_columns
    ]
    .head(50)
)


# ============================================================
# SAVE ROUND 2 STAGE 1 OUTPUTS
# ============================================================

STAGE1_AUDIT_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage1_audit.csv"
)

STAGE1_PASSED_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage1_passed.csv"
)

STAGE1_REJECTED_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage1_rejected.csv"
)

STAGE1_SUMMARY_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage1_class_summary.csv"
)


stage1_df.to_csv(
    STAGE1_AUDIT_PATH,
    index=False,
    encoding="utf-8"
)

stage1_passed_df.to_csv(
    STAGE1_PASSED_PATH,
    index=False,
    encoding="utf-8"
)

stage1_rejected_df.to_csv(
    STAGE1_REJECTED_PATH,
    index=False,
    encoding="utf-8"
)

stage1_class_summary.to_csv(
    STAGE1_SUMMARY_PATH,
    index=False,
    encoding="utf-8"
)


print(
    "\nRound 2 Stage 1 outputs saved."
)

print(
    "Audit:",
    STAGE1_AUDIT_PATH
)

print(
    "Passed:",
    STAGE1_PASSED_PATH
)

print(
    "Rejected:",
    STAGE1_REJECTED_PATH
)

print(
    "Summary:",
    STAGE1_SUMMARY_PATH
)

print(
    "\nSamples passed to Stage 2:",
    len(stage1_passed_df)
)

Round 2 Stage 1 input samples: 2603

Class distribution:


,Class,Count
0,Sexism,1269
1,Religious Hate,888
2,Profane,446



XLM-R checkpoint:
/home/jovyan/project work/data_analyssis/classifier/outputs/xlm_roberta_tuned/checkpoint-1203
Checkpoint exists: True


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Device: cuda
Frozen XLM-R verifier loaded successfully.
Prediction function loaded.

ROUND 2 - STAGE 1 LABEL CONSISTENCY
Input samples: 2603
Passed: 844
Rejected: 1759
Pass rate: 32.42%

Class-wise Stage 1 results:


...


Rejected samples only: target class × predicted class


xlmr_predicted_class,Abusive/Offensive,Normal,Profane,Religious Hate,Sexism
target_label,,,,,
Profane,114,100,0,2,12
Religious Hate,350,238,18,0,36
Sexism,612,208,53,16,0



Rejected samples: 1759


,production_candidate_id,class_id,target_label,generated_text,xlmr_predicted_class_id,xlmr_predicted_class,stage1_rejection_reason
0,class4_batch004_sample030600,4,Profane,"mera ghar ka sara kamaal kia tha, aaj mujhe ky...",1,Normal,Predicted class does not match intended class
1,class4_batch004_sample030601,4,Profane,bhai sahab ye duniya mein sabse kharabi galti ha,1,Normal,Predicted class does not match intended class
2,class4_batch004_sample030602,4,Profane,bhenchod ye 25 feb ko koi aisa nhi hota bharwe,0,Abusive/Offensive,Predicted class does not match intended class
3,class4_batch004_sample030603,4,Profane,teri maa ki phudi ka tatti gandu🖐️,0,Abusive/Offensive,Predicted class does not match intended class
4,class4_batch004_sample030607,4,Profane,tum logon ne apni maa ko mc bana diya hai😂,0,Abusive/Offensive,Predicted class does not match intended class
5,class4_batch004_sample030608,4,Profane,ye abhi tak aisa hi hua tha yaar?,1,Normal,Predicted class does not match intended class
6,class4_batch004_sample030609,4,Profane,"rt : ""yahodi yahan ka baap khanzeer khanzeer k...",0,Abusive/Offensive,Predicted class does not match intended class
7,class4_batch004_sample030611,4,Profane,pata nahi ab tumhari kya larkiyon ko goli marn...,0,Abusive/Offensive,Predicted class does not match intended class
8,class4_batch004_sample030612,4,Profane,mera charsi hai ye bhenchod,0,Abusive/Offensive,Predicted class does not match intended class
9,class4_batch004_sample030613,4,Profane,mn tweet krna tha aur apka tweet dekh leta phi...,0,Abusive/Offensive,Predicted class does not match intended class



Round 2 Stage 1 outputs saved.
Audit: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage1_audit.csv
Passed: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage1_passed.csv
Rejected: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage1_rejected.csv
Summary: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage1_class_summary.csv

Samples passed to Stage 2: 844


In [11]:
# ============================================================
# STAGE 1
# TARGET CLASS × XLM-R PREDICTED CLASS
# ============================================================

stage1_confusion = pd.crosstab(
    stage1_df[
        "target_label"
    ],
    stage1_df[
        "xlmr_predicted_class"
    ],
    margins=True,
)

display(stage1_confusion)

xlmr_predicted_class,Abusive/Offensive,Normal,Profane,Religious Hate,Sexism,All
target_label,,,,,,
Profane,114,100,218,2,12,446
Religious Hate,350,238,18,246,36,888
Sexism,612,208,53,16,380,1269
All,1076,546,289,264,428,2603


In [12]:
# ============================================================
# STAGE 1
# REJECTION PATTERNS
# ============================================================

if len(stage1_rejected_df) > 0:

    stage1_rejection_patterns = pd.crosstab(
        stage1_rejected_df[
            "target_label"
        ],
        stage1_rejected_df[
            "xlmr_predicted_class"
        ],
    )

    display(
        stage1_rejection_patterns
    )

xlmr_predicted_class,Abusive/Offensive,Normal,Profane,Religious Hate,Sexism
target_label,,,,,
Profane,114,100,0,2,12
Religious Hate,350,238,18,0,36
Sexism,612,208,53,16,0


In [15]:
# ============================================================
# ROUND 2 - STAGE 2
# SETUP AUTHENTIC RUHSOLD REFERENCE DATA
# ============================================================

import numpy as np
import pandas as pd
import torch
import faiss

from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# Authentic RUHSOLD training data
# ------------------------------------------------------------

RUHSOLD_TRAIN_PATH = Path(
    "/home/jovyan/project work/data_analyssis/RUHSOLD_train.tsv"
)

ruhsold_train_df = pd.read_csv(
    RUHSOLD_TRAIN_PATH,
    sep="\t",
    names=[
        "text",
        "label",
    ],
)

print(
    "Authentic RUHSOLD training samples:",
    len(ruhsold_train_df)
)


# ------------------------------------------------------------
# Frozen E5 embedding model
# ------------------------------------------------------------

E5_MODEL_NAME = (
    "intfloat/multilingual-e5-large"
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

e5_model = SentenceTransformer(
    E5_MODEL_NAME,
    device=device,
)

print(
    "Loaded:",
    E5_MODEL_NAME
)

print(
    "Device:",
    device
)

Authentic RUHSOLD training samples: 6408


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loaded: intfloat/multilingual-e5-large
Device: cuda


In [16]:
# ============================================================
# ROUND 2 - STAGE 2
# BUILD AUTHENTIC RUHSOLD CLASS-CONDITIONAL FAISS INDEXES
# ============================================================

AUGMENTATION_CLASS_IDS = [
    2,
    3,
    4,
]

real_embeddings_by_class = {}
real_texts_by_class = {}
faiss_indexes_by_class = {}


for class_id in AUGMENTATION_CLASS_IDS:

    class_name = (
        ID_TO_LABEL[
            class_id
        ]
    )

    class_df = (
        ruhsold_train_df[
            ruhsold_train_df[
                "label"
            ] == class_id
        ]
        .copy()
        .reset_index(drop=True)
    )

    real_texts = (
        class_df[
            "text"
        ]
        .fillna("")
        .astype(str)
        .tolist()
    )

    # --------------------------------------------------------
    # SAME E5 formatting used in Round 1
    # --------------------------------------------------------

    texts_for_embedding = [
        "query: " + text
        for text in real_texts
    ]

    embeddings = (
        e5_model.encode(
            texts_for_embedding,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        .astype(
            np.float32
        )
    )

    real_embeddings_by_class[
        class_id
    ] = embeddings

    real_texts_by_class[
        class_id
    ] = real_texts

    # --------------------------------------------------------
    # Inner product = cosine similarity
    # because embeddings are normalized
    # --------------------------------------------------------

    embedding_dim = (
        embeddings.shape[1]
    )

    index = faiss.IndexFlatIP(
        embedding_dim
    )

    index.add(
        embeddings
    )

    faiss_indexes_by_class[
        class_id
    ] = index

    print(
        f"{class_id} - {class_name}: "
        f"{len(real_texts)} authentic samples, "
        f"{index.ntotal} vectors indexed"
    )


print(
    "\nAvailable FAISS indexes:"
)

print(
    sorted(
        faiss_indexes_by_class.keys()
    )
)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

2 - Religious Hate: 500 authentic samples, 500 vectors indexed


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

3 - Sexism: 537 authentic samples, 537 vectors indexed


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

4 - Profane: 411 authentic samples, 411 vectors indexed

Available FAISS indexes:
[2, 3, 4]


In [17]:
# ============================================================
# ROUND 2 - STAGE 2 INPUT
# ============================================================

stage2_input_df = pd.read_csv(
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage1_passed.csv"
)

stage2_input_df = (
    stage2_input_df
    .reset_index(drop=True)
)


print("=" * 60)
print(
    "ROUND 2 - STAGE 2 INPUT"
)
print("=" * 60)

print(
    "Stage 2 input samples:",
    len(stage2_input_df)
)

print(
    "\nClass distribution:"
)

display(
    stage2_input_df[
        "target_label"
    ]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

ROUND 2 - STAGE 2 INPUT
Stage 2 input samples: 844

Class distribution:


,Class,Count
0,Sexism,380
1,Religious Hate,246
2,Profane,218


In [18]:
# ============================================================
# ROUND 2 - STAGE 2
# ENCODE STAGE 1 SURVIVORS
# ============================================================

synthetic_texts_e5 = [
    "query: " + str(text)
    for text in stage2_input_df[
        "generated_text"
    ].tolist()
]


synthetic_embeddings = (
    e5_model.encode(
        synthetic_texts_e5,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    .astype(
        np.float32
    )
)


print(
    "Synthetic embedding shape:",
    synthetic_embeddings.shape
)

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Synthetic embedding shape: (844, 1024)


In [19]:
# ============================================================
# ROUND 2 - STAGE 2
# SAME-CLASS AUTHENTIC SEMANTIC RETRIEVAL
# ============================================================

K = 10

stage2_rows = []


for row_position, row in (
    stage2_input_df.iterrows()
):

    target_class_id = int(
        row[
            "class_id"
        ]
    )

    query_embedding = (
        synthetic_embeddings[
            row_position
        ]
        .reshape(
            1,
            -1,
        )
    )

    index = (
        faiss_indexes_by_class[
            target_class_id
        ]
    )

    scores, indices = (
        index.search(
            query_embedding,
            K,
        )
    )

    scores = scores[0]
    indices = indices[0]

    nearest_texts = [
        real_texts_by_class[
            target_class_id
        ][idx]
        for idx in indices
    ]

    result = (
        row.to_dict()
    )

    result.update(
        {
            "stage2_max_similarity":
                float(
                    scores[0]
                ),

            "stage2_mean_top10_similarity":
                float(
                    scores.mean()
                ),

            "stage2_nearest_real_text":
                nearest_texts[0],

            "stage2_nearest_real_similarity":
                float(
                    scores[0]
                ),
        }
    )

    stage2_rows.append(
        result
    )


stage2_similarity_df = (
    pd.DataFrame(
        stage2_rows
    )
)


print(
    "Stage 2 semantic retrieval completed:"
)

print(
    len(
        stage2_similarity_df
    ),
    "samples"
)

Stage 2 semantic retrieval completed:
844 samples


In [20]:
# ============================================================
# ROUND 2 - STAGE 2
# APPLY FROZEN SEMANTIC THRESHOLDS
# ============================================================

LOWER_SIMILARITY_THRESHOLD = 0.85
UPPER_SIMILARITY_THRESHOLD = 0.98


stage2_final_df = (
    stage2_similarity_df
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Acceptance rule:
#
# 0.85 <= max same-class similarity < 0.98
# ------------------------------------------------------------

stage2_final_df[
    "stage2_pass"
] = (
    (
        stage2_final_df[
            "stage2_max_similarity"
        ]
        >= LOWER_SIMILARITY_THRESHOLD
    )
    &
    (
        stage2_final_df[
            "stage2_max_similarity"
        ]
        < UPPER_SIMILARITY_THRESHOLD
    )
)


# ------------------------------------------------------------
# Rejection reason
# ------------------------------------------------------------

stage2_final_df[
    "stage2_rejection_reason"
] = pd.NA


stage2_final_df.loc[
    stage2_final_df[
        "stage2_max_similarity"
    ] < LOWER_SIMILARITY_THRESHOLD,
    "stage2_rejection_reason"
] = (
    "Below semantic similarity threshold"
)


stage2_final_df.loc[
    stage2_final_df[
        "stage2_max_similarity"
    ] >= UPPER_SIMILARITY_THRESHOLD,
    "stage2_rejection_reason"
] = (
    "Potential duplicate / near-duplicate"
)


# ============================================================
# PASSED / REJECTED DATASETS
# ============================================================

stage2_passed_df = (
    stage2_final_df[
        stage2_final_df[
            "stage2_pass"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


stage2_rejected_df = (
    stage2_final_df[
        ~stage2_final_df[
            "stage2_pass"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

In [21]:
# ============================================================
# ROUND 2 - STAGE 2 SUMMARY
# ============================================================

print("\n" + "=" * 60)

print(
    "ROUND 2 - STAGE 2 "
    "SEMANTIC PLAUSIBILITY + "
    "REAL-DATA NEAR-DUPLICATE FILTER"
)

print("=" * 60)


print(
    "Input samples:",
    len(stage2_final_df)
)

print(
    "Passed:",
    len(stage2_passed_df)
)

print(
    "Rejected:",
    len(stage2_rejected_df)
)

print(
    "Pass rate:",
    f"{len(stage2_passed_df) / len(stage2_final_df):.2%}"
)


print(
    "\nFrozen thresholds:"
)

print(
    "Lower similarity threshold:",
    LOWER_SIMILARITY_THRESHOLD
)

print(
    "Upper similarity threshold:",
    UPPER_SIMILARITY_THRESHOLD
)


# ============================================================
# CLASS-WISE SUMMARY
# ============================================================

stage2_class_summary = (
    stage2_final_df
    .groupby(
        [
            "class_id",
            "target_label",
        ],
        dropna=False,
    )
    .agg(
        input_samples=(
            "stage2_pass",
            "size",
        ),
        passed=(
            "stage2_pass",
            "sum",
        ),
    )
    .reset_index()
)


stage2_class_summary[
    "rejected"
] = (
    stage2_class_summary[
        "input_samples"
    ]
    -
    stage2_class_summary[
        "passed"
    ]
)


stage2_class_summary[
    "pass_rate"
] = (
    stage2_class_summary[
        "passed"
    ]
    /
    stage2_class_summary[
        "input_samples"
    ]
)


print(
    "\nClass-wise Stage 2 results:"
)

display(
    stage2_class_summary.round(4)
)


# ============================================================
# REJECTION REASONS
# ============================================================

print(
    "\nRejection reasons:"
)

display(
    stage2_rejected_df[
        "stage2_rejection_reason"
    ]
    .value_counts()
    .rename_axis("Reason")
    .reset_index(name="Count")
)


ROUND 2 - STAGE 2 SEMANTIC PLAUSIBILITY + REAL-DATA NEAR-DUPLICATE FILTER
Input samples: 844
Passed: 832
Rejected: 12
Pass rate: 98.58%

Frozen thresholds:
Lower similarity threshold: 0.85
Upper similarity threshold: 0.98

Class-wise Stage 2 results:


,class_id,target_label,input_samples,passed,rejected,pass_rate
0,2,Religious Hate,246,244,2,0.9919
1,3,Sexism,380,377,3,0.9921
2,4,Profane,218,211,7,0.9679



Rejection reasons:


,Reason,Count
0,Potential duplicate / near-duplicate,10
1,Below semantic similarity threshold,2


In [22]:
# ============================================================
# SAVE ROUND 2 STAGE 2 OUTPUTS
# ============================================================

STAGE2_AUDIT_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage2_audit.csv"
)

STAGE2_PASSED_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage2_passed.csv"
)

STAGE2_REJECTED_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage2_rejected.csv"
)

STAGE2_SUMMARY_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage2_class_summary.csv"
)


stage2_final_df.to_csv(
    STAGE2_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

stage2_passed_df.to_csv(
    STAGE2_PASSED_PATH,
    index=False,
    encoding="utf-8",
)

stage2_rejected_df.to_csv(
    STAGE2_REJECTED_PATH,
    index=False,
    encoding="utf-8",
)

stage2_class_summary.to_csv(
    STAGE2_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)


print(
    "\nRound 2 Stage 2 outputs saved."
)

print(
    "Audit:",
    STAGE2_AUDIT_PATH
)

print(
    "Passed:",
    STAGE2_PASSED_PATH
)

print(
    "Rejected:",
    STAGE2_REJECTED_PATH
)

print(
    "Summary:",
    STAGE2_SUMMARY_PATH
)

print(
    "\nSamples passed to Stage 3:",
    len(stage2_passed_df)
)


Round 2 Stage 2 outputs saved.
Audit: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage2_audit.csv
Passed: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage2_passed.csv
Rejected: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage2_rejected.csv
Summary: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage2_class_summary.csv

Samples passed to Stage 3: 832


In [23]:
# ============================================================
# ROUND 2 QC
# STAGE 3 - SYNTHETIC-TO-SYNTHETIC DIVERSITY FILTER
#
# Compare Round 2 survivors against:
#   1. Existing Round 1 accepted synthetic bank
#   2. Earlier accepted Round 2 samples
#
# Frozen near-duplicate threshold = 0.98
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROUND1_SYNTHETIC_BANK_PATH = (
    GENERATION_DIR
    / "qc_pipeline"
    / "full_production"
    / "accepted_synthetic_bank.csv"
)

ROUND2_STAGE2_PASSED_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage2_passed.csv"
)


print(
    "Round 1 synthetic bank exists:",
    ROUND1_SYNTHETIC_BANK_PATH.exists()
)

print(
    "Round 2 Stage 2 survivors exist:",
    ROUND2_STAGE2_PASSED_PATH.exists()
)

Round 1 synthetic bank exists: True
Round 2 Stage 2 survivors exist: True


In [24]:
# ============================================================
# LOAD EXISTING ROUND 1 BANK
# AND ROUND 2 STAGE 2 SURVIVORS
# ============================================================

existing_bank_df = pd.read_csv(
    ROUND1_SYNTHETIC_BANK_PATH
)

round2_stage3_input_df = pd.read_csv(
    ROUND2_STAGE2_PASSED_PATH
)

existing_bank_df = (
    existing_bank_df
    .copy()
    .reset_index(drop=True)
)

round2_stage3_input_df = (
    round2_stage3_input_df
    .copy()
    .reset_index(drop=True)
)


print("=" * 70)
print("ROUND 2 - STAGE 3 INPUT")
print("=" * 70)

print(
    "Existing Round 1 synthetic bank:",
    len(existing_bank_df)
)

print(
    "Round 2 Stage 2 survivors:",
    len(round2_stage3_input_df)
)


print(
    "\nExisting bank class distribution:"
)

display(
    existing_bank_df[
        "target_label"
    ]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)


print(
    "\nRound 2 Stage 3 input distribution:"
)

display(
    round2_stage3_input_df[
        "target_label"
    ]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

ROUND 2 - STAGE 3 INPUT
Existing Round 1 synthetic bank: 751
Round 2 Stage 2 survivors: 832

Existing bank class distribution:


,Class,Count
0,Religious Hate,279
1,Profane,253
2,Sexism,219



Round 2 Stage 3 input distribution:


,Class,Count
0,Sexism,377
1,Religious Hate,244
2,Profane,211


In [25]:
# ============================================================
# ROUND 2 - STAGE 3
# ENCODE EXISTING BANK + ROUND 2 CANDIDATES
# ============================================================

existing_bank_texts_e5 = [
    "query: " + str(text)
    for text in existing_bank_df[
        "generated_text"
    ].tolist()
]

round2_candidate_texts_e5 = [
    "query: " + str(text)
    for text in round2_stage3_input_df[
        "generated_text"
    ].tolist()
]


existing_bank_embeddings = (
    e5_model.encode(
        existing_bank_texts_e5,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    .astype(np.float32)
)


round2_candidate_embeddings = (
    e5_model.encode(
        round2_candidate_texts_e5,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    .astype(np.float32)
)


print(
    "Existing bank embedding shape:",
    existing_bank_embeddings.shape
)

print(
    "Round 2 candidate embedding shape:",
    round2_candidate_embeddings.shape
)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Existing bank embedding shape: (751, 1024)
Round 2 candidate embedding shape: (832, 1024)


In [26]:
# ============================================================
# ROUND 2 - STAGE 3
# CUMULATIVE SYNTHETIC DIVERSITY FILTER
# ============================================================

SYNTHETIC_DUPLICATE_THRESHOLD = 0.98


round2_stage3_final_df = (
    round2_stage3_input_df
    .copy()
    .reset_index(drop=True)
)


round2_stage3_final_df[
    "stage3_pass"
] = True

round2_stage3_final_df[
    "stage3_rejection_reason"
] = pd.NA

round2_stage3_final_df[
    "stage3_duplicate_of"
] = pd.NA

round2_stage3_final_df[
    "stage3_duplicate_similarity"
] = np.nan

round2_stage3_final_df[
    "stage3_duplicate_source"
] = pd.NA


# ============================================================
# BUILD INITIAL ACCEPTED BANK
# ============================================================

# We keep dataframe rows and embeddings together.
accepted_bank_df = (
    existing_bank_df
    .copy()
    .reset_index(drop=True)
)

accepted_bank_embeddings = (
    existing_bank_embeddings
    .copy()
)


print(
    "Initial accepted bank size:",
    len(accepted_bank_df)
)

Initial accepted bank size: 751


In [27]:
# ============================================================
# PROCESS ROUND 2 CANDIDATES SEQUENTIALLY
# ============================================================

for current_pos, row in (
    round2_stage3_final_df.iterrows()
):

    target_class_id = int(
        row["class_id"]
    )

    current_embedding = (
        round2_candidate_embeddings[
            current_pos
        ]
    )

    # --------------------------------------------------------
    # Compare only against SAME-CLASS accepted synthetics
    # --------------------------------------------------------

    same_class_positions = (
        accepted_bank_df.index[
            accepted_bank_df[
                "class_id"
            ].astype(int)
            == target_class_id
        ]
        .tolist()
    )

    # Should not normally happen, but handle safely.
    if len(same_class_positions) == 0:

        accepted_bank_df = pd.concat(
            [
                accepted_bank_df,
                pd.DataFrame(
                    [row.to_dict()]
                ),
            ],
            ignore_index=True,
        )

        accepted_bank_embeddings = np.vstack(
            [
                accepted_bank_embeddings,
                current_embedding.reshape(1, -1),
            ]
        )

        continue

    # --------------------------------------------------------
    # Same-class accepted embeddings
    # --------------------------------------------------------

    same_class_embeddings = (
        accepted_bank_embeddings[
            same_class_positions
        ]
    )

    # Since embeddings are normalized,
    # matrix multiplication = cosine similarity.
    similarities = np.dot(
        same_class_embeddings,
        current_embedding,
    )

    max_local_position = int(
        np.argmax(
            similarities
        )
    )

    max_similarity = float(
        similarities[
            max_local_position
        ]
    )

    matched_bank_position = (
        same_class_positions[
            max_local_position
        ]
    )

    matched_row = (
        accepted_bank_df.iloc[
            matched_bank_position
        ]
    )

    # --------------------------------------------------------
    # Reject near-duplicate
    # --------------------------------------------------------

    if (
        max_similarity
        >= SYNTHETIC_DUPLICATE_THRESHOLD
    ):

        round2_stage3_final_df.loc[
            current_pos,
            "stage3_pass",
        ] = False

        round2_stage3_final_df.loc[
            current_pos,
            "stage3_rejection_reason",
        ] = (
            "Synthetic near-duplicate"
        )

        # Save matched candidate ID if available.
        duplicate_id = (
            matched_row[
                "production_candidate_id"
            ]
            if (
                "production_candidate_id"
                in matched_row.index
            )
            else matched_bank_position
        )

        round2_stage3_final_df.loc[
            current_pos,
            "stage3_duplicate_of",
        ] = duplicate_id

        round2_stage3_final_df.loc[
            current_pos,
            "stage3_duplicate_similarity",
        ] = max_similarity

        # Determine whether match came from Round 1
        # or from a Round 2 sample accepted earlier.
        if (
            matched_bank_position
            < len(existing_bank_df)
        ):

            duplicate_source = (
                "Round 1 accepted synthetic bank"
            )

        else:

            duplicate_source = (
                "Earlier accepted Round 2 synthetic"
            )

        round2_stage3_final_df.loc[
            current_pos,
            "stage3_duplicate_source",
        ] = duplicate_source

    # --------------------------------------------------------
    # Otherwise ACCEPT and immediately add to synthetic bank
    # --------------------------------------------------------

    else:

        accepted_row = (
            row.to_dict()
        )

        accepted_bank_df = pd.concat(
            [
                accepted_bank_df,
                pd.DataFrame(
                    [accepted_row]
                ),
            ],
            ignore_index=True,
        )

        accepted_bank_embeddings = np.vstack(
            [
                accepted_bank_embeddings,
                current_embedding.reshape(
                    1,
                    -1,
                ),
            ]
        )

In [28]:
# ============================================================
# ROUND 2 STAGE 3
# CREATE PASSED / REJECTED DATASETS
# ============================================================

round2_stage3_passed_df = (
    round2_stage3_final_df[
        round2_stage3_final_df[
            "stage3_pass"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


round2_stage3_rejected_df = (
    round2_stage3_final_df[
        ~round2_stage3_final_df[
            "stage3_pass"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

In [29]:
# ============================================================
# ROUND 2 - STAGE 3 SUMMARY
# ============================================================

print("\n" + "=" * 70)

print(
    "ROUND 2 - STAGE 3 "
    "SYNTHETIC DIVERSITY FILTERING"
)

print("=" * 70)

print(
    "Existing bank before Round 2:",
    len(existing_bank_df)
)

print(
    "Round 2 input samples:",
    len(round2_stage3_final_df)
)

print(
    "Round 2 passed:",
    len(round2_stage3_passed_df)
)

print(
    "Round 2 rejected:",
    len(round2_stage3_rejected_df)
)

print(
    "Round 2 Stage 3 pass rate:",
    f"{len(round2_stage3_passed_df) / len(round2_stage3_final_df):.2%}"
)

print(
    "Near-duplicate threshold:",
    SYNTHETIC_DUPLICATE_THRESHOLD
)

print(
    "\nFinal cumulative synthetic bank size:",
    len(accepted_bank_df)
)


ROUND 2 - STAGE 3 SYNTHETIC DIVERSITY FILTERING
Existing bank before Round 2: 751
Round 2 input samples: 832
Round 2 passed: 795
Round 2 rejected: 37
Round 2 Stage 3 pass rate: 95.55%
Near-duplicate threshold: 0.98

Final cumulative synthetic bank size: 1546


In [30]:
# ============================================================
# ROUND 2 STAGE 3
# CLASS-WISE SUMMARY
# ============================================================

round2_stage3_class_summary = (
    round2_stage3_final_df
    .groupby(
        [
            "class_id",
            "target_label",
        ]
    )
    .agg(
        input_samples=(
            "stage3_pass",
            "size",
        ),
        passed=(
            "stage3_pass",
            "sum",
        ),
    )
    .reset_index()
)


round2_stage3_class_summary[
    "rejected"
] = (
    round2_stage3_class_summary[
        "input_samples"
    ]
    -
    round2_stage3_class_summary[
        "passed"
    ]
)


round2_stage3_class_summary[
    "pass_rate"
] = (
    round2_stage3_class_summary[
        "passed"
    ]
    /
    round2_stage3_class_summary[
        "input_samples"
    ]
)


print(
    "\nClass-wise Round 2 Stage 3 results:"
)

display(
    round2_stage3_class_summary.round(4)
)


Class-wise Round 2 Stage 3 results:


,class_id,target_label,input_samples,passed,rejected,pass_rate
0,2,Religious Hate,244,232,12,0.9508
1,3,Sexism,377,357,20,0.9469
2,4,Profane,211,206,5,0.9763


In [31]:
# ============================================================
# ROUND 2 STAGE 3
# REJECTION SOURCE SUMMARY
# ============================================================

print(
    "\nRejection reasons:"
)

display(
    round2_stage3_rejected_df[
        "stage3_rejection_reason"
    ]
    .value_counts()
    .rename_axis("Reason")
    .reset_index(name="Count")
)


print(
    "\nDuplicate source:"
)

display(
    round2_stage3_rejected_df[
        "stage3_duplicate_source"
    ]
    .value_counts()
    .rename_axis("Matched against")
    .reset_index(name="Count")
)


Rejection reasons:


,Reason,Count
0,Synthetic near-duplicate,37



Duplicate source:


,Matched against,Count
0,Round 1 accepted synthetic bank,29
1,Earlier accepted Round 2 synthetic,8


In [32]:
# ============================================================
# SAVE ROUND 2 STAGE 3 OUTPUTS
# ============================================================

ROUND2_STAGE3_AUDIT_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage3_audit.csv"
)

ROUND2_STAGE3_PASSED_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage3_passed.csv"
)

ROUND2_STAGE3_REJECTED_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage3_rejected.csv"
)

ROUND2_STAGE3_SUMMARY_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "round2_stage3_class_summary.csv"
)

FINAL_CUMULATIVE_BANK_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "accepted_synthetic_bank_round1_plus_round2.csv"
)


round2_stage3_final_df.to_csv(
    ROUND2_STAGE3_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

round2_stage3_passed_df.to_csv(
    ROUND2_STAGE3_PASSED_PATH,
    index=False,
    encoding="utf-8",
)

round2_stage3_rejected_df.to_csv(
    ROUND2_STAGE3_REJECTED_PATH,
    index=False,
    encoding="utf-8",
)

round2_stage3_class_summary.to_csv(
    ROUND2_STAGE3_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)

accepted_bank_df.to_csv(
    FINAL_CUMULATIVE_BANK_PATH,
    index=False,
    encoding="utf-8",
)


print(
    "\nRound 2 Stage 3 outputs saved."
)

print(
    "Audit:",
    ROUND2_STAGE3_AUDIT_PATH
)

print(
    "Passed:",
    ROUND2_STAGE3_PASSED_PATH
)

print(
    "Rejected:",
    ROUND2_STAGE3_REJECTED_PATH
)

print(
    "Summary:",
    ROUND2_STAGE3_SUMMARY_PATH
)

print(
    "Final cumulative bank:",
    FINAL_CUMULATIVE_BANK_PATH
)


Round 2 Stage 3 outputs saved.
Audit: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage3_audit.csv
Passed: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage3_passed.csv
Rejected: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage3_rejected.csv
Summary: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/round2_stage3_class_summary.csv
Final cumulative bank: /home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/accepted_synthetic_bank_round1_plus_round2.csv


In [33]:
# ============================================================
# CHECK FINAL BANK AGAINST 1x AUGMENTATION TARGETS
# ============================================================

TARGET_SYNTHETIC_COUNTS = {
    2: 500,   # Religious Hate
    3: 537,   # Sexism
    4: 411,   # Profane
}


final_bank_counts = (
    accepted_bank_df[
        "class_id"
    ]
    .astype(int)
    .value_counts()
    .sort_index()
)


print("=" * 70)

print(
    "FINAL SYNTHETIC BANK VS 1x TARGET"
)

print("=" * 70)


for class_id, target_count in (
    TARGET_SYNTHETIC_COUNTS.items()
):

    actual_count = int(
        final_bank_counts.get(
            class_id,
            0,
        )
    )

    difference = (
        actual_count
        - target_count
    )

    print(
        f"{class_id} - "
        f"{ID_TO_LABEL[class_id]}: "
        f"{actual_count} / {target_count} "
        f"({difference:+d})"
    )

FINAL SYNTHETIC BANK VS 1x TARGET
2 - Religious Hate: 511 / 500 (+11)
3 - Sexism: 576 / 537 (+39)
4 - Profane: 459 / 411 (+48)


In [34]:
# ============================================================
# BUILD FINAL 1x SYNTHETIC AUGMENTATION SET
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# LOAD FINAL CUMULATIVE ACCEPTED BANK
# ============================================================

FINAL_CUMULATIVE_BANK_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "accepted_synthetic_bank_round1_plus_round2.csv"
)

accepted_bank_df = pd.read_csv(
    FINAL_CUMULATIVE_BANK_PATH
)

print(
    "Cumulative accepted synthetic samples:",
    len(accepted_bank_df)
)

print(
    "\nAvailable class distribution:"
)

display(
    accepted_bank_df[
        [
            "class_id",
            "target_label",
        ]
    ]
    .value_counts()
    .rename("count")
    .reset_index()
)


# ============================================================
# FROZEN 1x TARGET COUNTS
# ============================================================

TARGET_SYNTHETIC_COUNTS = {
    2: 500,   # Religious Hate
    3: 537,   # Sexism
    4: 411,   # Profane
}

FINAL_SELECTION_SEED = 42


# ============================================================
# SAMPLE EXACT TARGET COUNT FROM EACH CLASS
# ============================================================

final_synthetic_parts = []


for class_id, target_count in (
    TARGET_SYNTHETIC_COUNTS.items()
):

    class_df = (
        accepted_bank_df[
            accepted_bank_df[
                "class_id"
            ].astype(int) == class_id
        ]
        .copy()
        .reset_index(drop=True)
    )

    available_count = len(
        class_df
    )

    print(
        f"{class_id} - {ID_TO_LABEL[class_id]}:"
    )

    print(
        f"Available: {available_count}"
    )

    print(
        f"Required : {target_count}"
    )

    if available_count < target_count:

        raise ValueError(
            f"Not enough accepted synthetic samples "
            f"for class {class_id}."
        )

    selected_df = (
        class_df
        .sample(
            n=target_count,
            random_state=FINAL_SELECTION_SEED,
            replace=False,
        )
        .copy()
        .reset_index(drop=True)
    )

    final_synthetic_parts.append(
        selected_df
    )


# ============================================================
# COMBINE FINAL 1x SYNTHETIC SET
# ============================================================

final_1x_synthetic_df = pd.concat(
    final_synthetic_parts,
    ignore_index=True
)


print(
    "\nFinal 1x synthetic dataset shape:",
    final_1x_synthetic_df.shape
)

print(
    "\nFinal synthetic class distribution:"
)

display(
    final_1x_synthetic_df[
        [
            "class_id",
            "target_label",
        ]
    ]
    .value_counts()
    .rename("count")
    .reset_index()
)


# ============================================================
# SANITY CHECKS
# ============================================================

assert (
    len(final_1x_synthetic_df)
    == sum(
        TARGET_SYNTHETIC_COUNTS.values()
    )
)

for class_id, expected_count in (
    TARGET_SYNTHETIC_COUNTS.items()
):

    actual_count = int(
        (
            final_1x_synthetic_df[
                "class_id"
            ].astype(int)
            == class_id
        ).sum()
    )

    assert actual_count == expected_count


print(
    "\nFinal 1x synthetic dataset verified successfully."
)

Cumulative accepted synthetic samples: 1546

Available class distribution:


,class_id,target_label,count
0,3,Sexism,576
1,2,Religious Hate,511
2,4,Profane,459


2 - Religious Hate:
Available: 511
Required : 500
3 - Sexism:
Available: 576
Required : 537
4 - Profane:
Available: 459
Required : 411

Final 1x synthetic dataset shape: (1448, 42)

Final synthetic class distribution:


,class_id,target_label,count
0,3,Sexism,537
1,2,Religious Hate,500
2,4,Profane,411



Final 1x synthetic dataset verified successfully.


In [35]:
# ============================================================
# SAVE FINAL 1x SYNTHETIC DATASET
# ============================================================

FINAL_1X_SYNTHETIC_PATH = (
    ROUND2_QC_OUTPUT_DIR
    / "final_1x_synthetic_training_set.csv"
)

final_1x_synthetic_df.to_csv(
    FINAL_1X_SYNTHETIC_PATH,
    index=False,
    encoding="utf-8",
)

print(
    "Saved final 1x synthetic dataset to:"
)

print(
    FINAL_1X_SYNTHETIC_PATH
)

Saved final 1x synthetic dataset to:
/home/jovyan/project work/data_analyssis/fine tuning/outputs/synthetic_generation/qc_pipeline/round2/final_1x_synthetic_training_set.csv
